####Setup

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


catalog = "crypto_lakehouse"
bronze_schema = "bronze"
silver_schema = "silver"

display(
    spark.sql(
        f"SHOW TABLES IN {catalog}.{bronze_schema}"
    )
)

database,tableName,isTemporary
bronze,binance_account_snapshots,false
bronze,binance_convert_trades,false
bronze,binance_deposits,false
bronze,binance_funding_account_snapshots,false
bronze,binance_p2p_trades,false
bronze,binance_price_history,false
bronze,binance_spot_trades,false
bronze,binance_transfers,false
bronze,binance_withdrawals,false
bronze,coindcx_balances,false


####Giottus spot balance

In [0]:
balances_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.giottus_spot_balances"
)

print("Bronze balance rows:", balances_bronze_df.count())
balances_bronze_df.printSchema()


Bronze balance rows: 1
root
 |-- asset: string (nullable = true)
 |-- free: string (nullable = true)
 |-- locked: string (nullable = true)
 |-- lockedFd: string (nullable = true)
 |-- lockedOtc: string (nullable = true)
 |-- lockedStaking: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- exchange: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [0]:
giottus_balances_silver_df = (
    balances_bronze_df
    .select(
        col("source_file_name").alias("snapshot_id"),
        lit(None).cast("string").alias("account_id"),
        upper(col("asset")).alias("asset"),
        col("free").cast("decimal(38,18)").alias("free_balance"),
        col("locked").cast("decimal(38,18)").alias("locked_balance"),
        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at")
    )
    .withColumn(
        "total_balance",
        col("free_balance") + col("locked_balance")
    )
    .withColumn("silver_processed_at", current_timestamp())
)

display(giottus_balances_silver_df)

snapshot_id,account_id,asset,free_balance,locked_balance,exchange,_source_file,bronze_ingested_at,total_balance,silver_processed_at
balances_20260925T182205_395071Z_3f93ddf0.json,null,INR,0.004980010000000000,0E-18,Giottus,s3://l1l2bucket/data/raw/giottus/balances/balances_20260925T182205_395071Z_3f93ddf0.json,2026-09-25T18:26:33.063Z,0.00498001000000000,2026-09-25T18:28:47.405Z


In [0]:
giottus_balances_final_df = (
    giottus_balances_silver_df
    .dropDuplicates(["snapshot_id", "asset", "account_id"])
)

(
    giottus_balances_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{silver_schema}.giottus_account_balances")
)

print(
    "Saved Giottus balance rows:",
    spark.table(f"{catalog}.{silver_schema}.giottus_account_balances").count()
)


Saved Giottus balance rows: 1


####Giottus trades

In [0]:
spot_trades_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.giottus_spot_trades"
)

ebes_trades_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.giottus_ebes_trades"
)

print("spot trade rows:", spot_trades_bronze_df.count())
print("ebes trade rows:", ebes_trades_bronze_df.count())

spot trade rows: 1
ebes trade rows: 1


In [0]:
giottus_trades_raw_df = (
    spot_trades_bronze_df
    .withColumn("trade_source", lit("Spot"))
    .unionByName(
        ebes_trades_bronze_df
        .withColumn("trade_source", lit("EBES")),
        allowMissingColumns=True
    )
)

print("Combined trade rows:", giottus_trades_raw_df.count())

display(giottus_trades_raw_df)

Combined trade rows: 2


symbol,id,orderId,price,qty,quoteQty,commission,commissionAsset,tds,time,isBuyer,isMaker,source_file_name,_source_file,exchange,source_type,ingested_at,trade_source
SOL/INR,228465533,1-128995378-1,6690.0,0.015058,100.73802,6e-05,SOL,0.000000,1781042089000,true,false,trades_20260925T182206_079516Z_4a735758.json,s3://l1l2bucket/data/raw/giottus/trades/SOLINR/trades_20260925T182206_079516Z_4a735758.json,Giottus,Spot_Trade,2026-09-25T18:26:45.897Z,Spot
SOL/INR,2858754,null,8556.674,0.0116,99.257,0.00004640,SOL,0.0000,1773925282000,true,false,ebes_trades_20260925T182206_856127Z_8f27aa2c.json,s3://l1l2bucket/data/raw/giottus/ebes_trades/SOLINR/ebes_trades_20260925T182206_856127Z_8f27aa2c.json,Giottus,Ebes_Trade,2026-09-25T18:26:58.434Z,EBES


In [0]:
giottus_trade_parts_df = (
    giottus_trades_raw_df
    .withColumn("base_asset", split(col("symbol"), "/")[0])
    .withColumn("quote_asset", split(col("symbol"), "/")[1])
    .withColumn(
        "side",
        when(col("isBuyer") == True, "BUY").otherwise("SELL")
    )
    .withColumn("trade_time", timestamp_millis(col("time")))
)

display(
    giottus_trade_parts_df.select(
        "trade_source","symbol","side",
        "base_asset","quote_asset",
        "trade_time"
    )
)

giottus_trade_parts_df.printSchema()

trade_source,symbol,side,base_asset,quote_asset,trade_time
Spot,SOL/INR,BUY,SOL,INR,2026-06-09T21:54:49.000Z
EBES,SOL/INR,BUY,SOL,INR,2026-03-19T13:01:22.000Z


root
 |-- symbol: string (nullable = true)
 |-- id: long (nullable = true)
 |-- orderId: string (nullable = true)
 |-- price: string (nullable = true)
 |-- qty: string (nullable = true)
 |-- quoteQty: string (nullable = true)
 |-- commission: string (nullable = true)
 |-- commissionAsset: string (nullable = true)
 |-- tds: string (nullable = true)
 |-- time: long (nullable = true)
 |-- isBuyer: boolean (nullable = true)
 |-- isMaker: boolean (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- exchange: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- trade_source: string (nullable = false)
 |-- base_asset: string (nullable = true)
 |-- quote_asset: string (nullable = true)
 |-- side: string (nullable = false)
 |-- trade_time: timestamp (nullable = true)



In [0]:
decimal_type = "decimal(38,18)"

giottus_trade_amounts_df = (
    giottus_trade_parts_df
    .withColumn(
        "asset_spent",
        when(col("side") == "BUY", col("quote_asset"))
        .otherwise(col("base_asset"))
    )
    .withColumn(
        "amount_spent",
        when(col("side") == "BUY", col("quoteQty"))
        .otherwise(col("qty"))
        .cast(decimal_type)
    )
    .withColumn(
        "asset_received",
        when(col("side") == "BUY", col("base_asset"))
        .otherwise(col("quote_asset"))
    )
    .withColumn(
        "amount_received",
        when(col("side") == "BUY", col("qty"))
        .otherwise(col("quoteQty"))
        .cast(decimal_type)
    )
)

display(
    giottus_trade_amounts_df.select(
        "trade_source", "side",
        "asset_spent", "amount_spent",
        "asset_received", "amount_received"
    )
)

trade_source,side,asset_spent,amount_spent,asset_received,amount_received
Spot,BUY,INR,100.738020000000000000,SOL,0.015058000000000000
EBES,BUY,INR,99.257000000000000000,SOL,0.011600000000000000


In [0]:
giottus_trades_silver_df = giottus_trade_amounts_df.select(
    col("id").cast("string").alias("transaction_id"),
    "trade_source",
    "symbol",
    "side",
    "asset_spent",
    "amount_spent",
    "asset_received",
    "amount_received",
    col("commission").cast(decimal_type).alias("fee_amount"),
    upper(col("commissionAsset")).alias("fee_asset"),
    "trade_time",
    "exchange",
    "_source_file",
    col("ingested_at").alias("bronze_ingested_at")
)

display(giottus_trades_silver_df)

transaction_id,trade_source,symbol,side,asset_spent,amount_spent,asset_received,amount_received,fee_amount,fee_asset,trade_time,exchange,_source_file,bronze_ingested_at
228465533,Spot,SOL/INR,BUY,INR,100.738020000000000000,SOL,0.015058000000000000,0.000060000000000000,SOL,2026-06-09T21:54:49.000Z,Giottus,s3://l1l2bucket/data/raw/giottus/trades/SOLINR/trades_20260925T182206_079516Z_4a735758.json,2026-09-25T18:26:45.897Z
2858754,EBES,SOL/INR,BUY,INR,99.257000000000000000,SOL,0.011600000000000000,0.000046400000000000,SOL,2026-03-19T13:01:22.000Z,Giottus,s3://l1l2bucket/data/raw/giottus/ebes_trades/SOLINR/ebes_trades_20260925T182206_856127Z_8f27aa2c.json,2026-09-25T18:26:58.434Z


In [0]:
from pyspark.sql.functions import col, concat_ws, current_timestamp, to_date

giottus_trades_final_df = (
    giottus_trades_silver_df
    .filter(
        col("transaction_id").isNotNull()
        & col("asset_spent").isNotNull()
        & col("asset_received").isNotNull()
        & (col("amount_spent") > 0)
        & (col("amount_received") > 0)
    )
    .withColumn(
        "trade_key",
        concat_ws(
            "|", "exchange", "trade_source",
            "symbol", "transaction_id"
        )
    )
    .dropDuplicates(["trade_key"])
    .withColumn("trade_date", to_date(col("trade_time")))
    .withColumn("silver_processed_at", current_timestamp())
)

print("Clean Giottus trades:", giottus_trades_final_df.count())
display(giottus_trades_final_df)

Clean Giottus trades: 2


transaction_id,trade_source,symbol,side,asset_spent,amount_spent,asset_received,amount_received,fee_amount,fee_asset,trade_time,exchange,_source_file,bronze_ingested_at,trade_key,trade_date,silver_processed_at
228465533,Spot,SOL/INR,BUY,INR,100.738020000000000000,SOL,0.015058000000000000,0.000060000000000000,SOL,2026-06-09T21:54:49.000Z,Giottus,s3://l1l2bucket/data/raw/giottus/trades/SOLINR/trades_20260925T182206_079516Z_4a735758.json,2026-09-25T18:26:45.897Z,Giottus|Spot|SOL/INR|228465533,2026-06-09,2026-09-25T18:29:04.983Z
2858754,EBES,SOL/INR,BUY,INR,99.257000000000000000,SOL,0.011600000000000000,0.000046400000000000,SOL,2026-03-19T13:01:22.000Z,Giottus,s3://l1l2bucket/data/raw/giottus/ebes_trades/SOLINR/ebes_trades_20260925T182206_856127Z_8f27aa2c.json,2026-09-25T18:26:58.434Z,Giottus|EBES|SOL/INR|2858754,2026-03-19,2026-09-25T18:29:04.983Z


In [0]:
(
    giottus_trades_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{catalog}.{silver_schema}.giottus_trades")
)

print(
    "Saved Giottus trades:",
    spark.table(f"{catalog}.{silver_schema}.giottus_trades").count()
)

Saved Giottus trades: 2


####Giottus fiat withdrawals

In [0]:
fiat_withdrawals_bronze_df = spark.table(
    f"{catalog}.bronze.giottus_fiat_withdrawals"
)

print("Fiat withdrawal rows:", fiat_withdrawals_bronze_df.count())
fiat_withdrawals_bronze_df.printSchema()

Fiat withdrawal rows: 0
root
 |-- action: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- fee: string (nullable = true)
 |-- status: string (nullable = true)
 |-- referenceId: string (nullable = true)
 |-- time: long (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- exchange: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [0]:
from pyspark.sql.functions import (
    col, current_timestamp, lit, timestamp_millis, upper
)

fiat_withdrawals_silver_df = (
    fiat_withdrawals_bronze_df
    .select(
        col("referenceId").alias("withdrawal_id"),
        lit("INR").alias("asset"),
        col("amount").cast("decimal(38,18)").alias("amount"),
        col("fee").cast("decimal(38,18)").alias("fee_amount"),
        upper(col("status")).alias("status"),
        timestamp_millis(col("time")).alias("withdrawal_time"),
        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at")
    )
    .withColumn("silver_processed_at", current_timestamp())
)

print("Clean fiat withdrawals:", fiat_withdrawals_silver_df.count())
fiat_withdrawals_silver_df.printSchema()

Clean fiat withdrawals: 0
root
 |-- withdrawal_id: string (nullable = true)
 |-- asset: string (nullable = false)
 |-- amount: decimal(38,18) (nullable = true)
 |-- fee_amount: decimal(38,18) (nullable = true)
 |-- status: string (nullable = true)
 |-- withdrawal_time: timestamp (nullable = true)
 |-- exchange: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- silver_processed_at: timestamp (nullable = false)



In [0]:
(
    fiat_withdrawals_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{catalog}.{silver_schema}.giottus_fiat_withdrawals")
)

print(
    "Saved fiat withdrawals:",
    spark.table(
        f"{catalog}.{silver_schema}.giottus_fiat_withdrawals"
    ).count()
)

Saved fiat withdrawals: 0


####Giottus fiat deposits

In [0]:
fiat_deposits_bronze_df = spark.table(
    f"{catalog}.bronze.giottus_fiat_deposits"
)

print("Fiat deposit rows:", fiat_deposits_bronze_df.count())
fiat_deposits_bronze_df.printSchema()

Fiat deposit rows: 2
root
 |-- action: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- fee: string (nullable = true)
 |-- status: string (nullable = true)
 |-- referenceId: string (nullable = true)
 |-- time: long (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- exchange: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [0]:
fiat_deposits_silver_df = (
    fiat_deposits_bronze_df
    .select(
        col("referenceId").alias("deposit_id"),
        lit("INR").alias("asset"),
        col("amount").cast("decimal(38,18)").alias("amount"),
        col("fee").cast("decimal(38,18)").alias("fee_amount"),
        upper(col("status")).alias("status"),
        timestamp_millis(col("time")).alias("deposit_time"),
        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at")
    )
    .withColumn("silver_processed_at", current_timestamp())
)

display(fiat_deposits_silver_df)

deposit_id,asset,amount,fee_amount,status,deposit_time,exchange,_source_file,bronze_ingested_at,silver_processed_at
391465032548,INR,100.000000000000000000,0E-18,SUCCESS,2026-06-09T01:32:53.000Z,Giottus,s3://l1l2bucket/data/raw/giottus/fiat_deposits/fiat_deposits_20260925T182208_433319Z_c4a5809e.json,2026-09-25T18:27:33.747Z,2026-09-25T18:29:20.850Z
792452268163,INR,100.000000000000000000,0E-18,SUCCESS,2026-03-18T20:26:49.000Z,Giottus,s3://l1l2bucket/data/raw/giottus/fiat_deposits/fiat_deposits_20260925T182208_433319Z_c4a5809e.json,2026-09-25T18:27:33.747Z,2026-09-25T18:29:20.850Z


In [0]:
fiat_deposits_final_df = (
    fiat_deposits_silver_df
    .dropDuplicates(["deposit_id"])
)

(
    fiat_deposits_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{catalog}.{silver_schema}.giottus_fiat_deposits")
)

print(
    "Saved fiat deposits:",
    spark.table(
        f"{catalog}.{silver_schema}.giottus_fiat_deposits"
    ).count()
)

Saved fiat deposits: 2


####Giottus crypto withdrawals

In [0]:
crypto_withdrawals_bronze_df = spark.table(
    f"{catalog}.bronze.giottus_crypto_withdrawals"
)

print(
    "Crypto withdrawal rows:",
    crypto_withdrawals_bronze_df.count()
)
crypto_withdrawals_bronze_df.printSchema()

Crypto withdrawal rows: 1
root
 |-- action: string (nullable = true)
 |-- asset: string (nullable = true)
 |-- cryptoAmount: string (nullable = true)
 |-- status: string (nullable = true)
 |-- transactionHash: string (nullable = true)
 |-- time: long (nullable = true)
 |-- network: string (nullable = true)
 |-- fromAddress: string (nullable = true)
 |-- toAddress: string (nullable = true)
 |-- fee: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- exchange: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [0]:
crypto_withdrawals_silver_df = (
    crypto_withdrawals_bronze_df
    .select(
        col("transactionHash").alias("withdrawal_id"),
        col("transactionHash").alias("transaction_hash"),
        upper(col("asset")).alias("asset"),
        col("cryptoAmount").cast("decimal(38,18)").alias("amount"),
        col("fee").cast("decimal(38,18)").alias("transaction_fee"),
        upper(col("network")).alias("network"),
        upper(col("status")).alias("status"),
        col("fromAddress").alias("from_address"),
        col("toAddress").alias("withdrawal_address"),
        timestamp_millis(col("time")).alias("withdrawal_time"),
        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at")
    )
    .withColumn("silver_processed_at", current_timestamp())
)

display(crypto_withdrawals_silver_df)

withdrawal_id,transaction_hash,asset,amount,transaction_fee,network,status,from_address,withdrawal_address,withdrawal_time,exchange,_source_file,bronze_ingested_at,silver_processed_at
6686245,6686245,SOL,0.025388000000000000,0.001200000000000000,SOL,SUCCESS,null,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,2026-06-29T03:29:59.000Z,Giottus,s3://l1l2bucket/data/raw/giottus/crypto_withdrawals/crypto_withdrawals_20260925T182207_918215Z_a652e68a.json,2026-09-25T18:27:22.087Z,2026-09-25T18:29:29.129Z


In [0]:
crypto_withdrawals_final_df = (
    crypto_withdrawals_silver_df
    .dropDuplicates(["withdrawal_id"])
)

(
    crypto_withdrawals_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{catalog}.{silver_schema}.giottus_crypto_withdrawals")
)

print(
    "Saved crypto withdrawals:",
    spark.table(
        f"{catalog}.{silver_schema}.giottus_crypto_withdrawals"
    ).count()
)

Saved crypto withdrawals: 1


####Giottus crypto deposits

In [0]:
crypto_deposits_bronze_df = spark.table(
    f"{catalog}.bronze.giottus_crypto_deposits"
)

print("Crypto deposit rows:", crypto_deposits_bronze_df.count())
crypto_deposits_bronze_df.printSchema()

Crypto deposit rows: 0
root
 |-- action: string (nullable = true)
 |-- asset: string (nullable = true)
 |-- cryptoAmount: string (nullable = true)
 |-- status: string (nullable = true)
 |-- transactionHash: string (nullable = true)
 |-- time: long (nullable = true)
 |-- network: string (nullable = true)
 |-- fromAddress: string (nullable = true)
 |-- toAddress: string (nullable = true)
 |-- fee: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- exchange: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [0]:
crypto_deposits_silver_df = (
    crypto_deposits_bronze_df
    .select(
        col("transactionHash").alias("deposit_id"),
        col("transactionHash").alias("transaction_hash"),
        upper(col("asset")).alias("asset"),
        col("cryptoAmount").cast("decimal(38,18)").alias("amount"),
        col("fee").cast("decimal(38,18)").alias("fee_amount"),
        upper(col("network")).alias("network"),
        upper(col("status")).alias("status"),
        col("fromAddress").alias("source_address"),
        col("toAddress").alias("deposit_address"),
        timestamp_millis(col("time")).alias("deposit_time"),
        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at")
    )
    .withColumn("silver_processed_at", current_timestamp())
)

print("Clean crypto deposits:", crypto_deposits_silver_df.count())
crypto_deposits_silver_df.printSchema()

Clean crypto deposits: 0
root
 |-- deposit_id: string (nullable = true)
 |-- transaction_hash: string (nullable = true)
 |-- asset: string (nullable = true)
 |-- amount: decimal(38,18) (nullable = true)
 |-- fee_amount: decimal(38,18) (nullable = true)
 |-- network: string (nullable = true)
 |-- status: string (nullable = true)
 |-- source_address: string (nullable = true)
 |-- deposit_address: string (nullable = true)
 |-- deposit_time: timestamp (nullable = true)
 |-- exchange: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- silver_processed_at: timestamp (nullable = false)



In [0]:
(
    crypto_deposits_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{catalog}.{silver_schema}.giottus_crypto_deposits")
)

print(
    "Saved crypto deposits:",
    spark.table(
        f"{catalog}.{silver_schema}.giottus_crypto_deposits"
    ).count()
)

Saved crypto deposits: 0
